<a href="https://colab.research.google.com/github/ffserro/analise-de-dados-boas-praticas/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import requests
from urllib.parse import quote
from google.colab import userdata


In [2]:
DATABRICKS_ACCOUNT_ID = 'dbc-c2b09e04-cc20'

# Exemplo: https://adb-1234567890123456.7.azuredatabricks.net
DATABRICKS_HOST = f'https://{DATABRICKS_ACCOUNT_ID}.cloud.databricks.com'

# Gere um PAT no Databricks e cole aqui
DATABRICKS_TOKEN = userdata.get('DATABRICKS_CLIENT_SECRET')

# Caminho do seu volume no Unity Catalog
VOLUME_PATH = "/Volumes/dados_fa/pessoal/org_xml"

# Opcional: diretório local no Colab para downloads
LOCAL_DOWNLOAD_DIR = "/content/databricks_xmls"

os.makedirs(LOCAL_DOWNLOAD_DIR, exist_ok=True)

In [3]:
session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {DATABRICKS_TOKEN}"
})

def _normalize_volume_path(path: str) -> str:
    path = path.strip()
    if not path.startswith("/Volumes/"):
        raise ValueError("Use um caminho começando com /Volumes/")
    return path

def list_volume_directory(volume_path: str):
    """
    Lista o conteúdo de um diretório em um Unity Catalog volume.
    """
    volume_path = _normalize_volume_path(volume_path)
    encoded_path = quote(volume_path, safe="/")
    url = f"{DATABRICKS_HOST}/api/2.0/fs/directories{encoded_path}"
    resp = session.get(url, timeout=60)
    resp.raise_for_status()
    return resp.json()

def download_volume_file(volume_file_path: str, local_path: str):
    """
    Baixa um arquivo de um Unity Catalog volume para o ambiente local do Colab.
    """
    volume_file_path = _normalize_volume_path(volume_file_path)
    encoded_path = quote(volume_file_path, safe="/")
    url = f"{DATABRICKS_HOST}/api/2.0/fs/files{encoded_path}"

    with session.get(url, stream=True, timeout=120) as resp:
        resp.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    return local_path

In [4]:
listing = list_volume_directory(VOLUME_PATH)
listing

{'contents': [{'path': '/Volumes/dados_fa/pessoal/org_xml/S02/',
   'is_directory': True,
   'name': 'S02'}]}

In [5]:
import requests

url = "https://dbc-c2b09e04-cc20.cloud.databricks.com/api/2.0/fs/directories/Volumes/dados_fa/pessoal/org_xml"
headers = {"Authorization": f"Bearer {DATABRICKS_TOKEN}"}

resp = requests.get(url, headers=headers, timeout=60)
print(resp.status_code)
print(resp.text)

200
{"contents":[{"path":"/Volumes/dados_fa/pessoal/org_xml/S02/","is_directory":true,"name":"S02"}]}
